# Fourier Neural Operator (FNO) basics

A neural operator learns a mapping between *functions*, not just between fixed-size vectors -
for example, "given this initial condition, predict the solution at time T" for a whole family
of initial conditions, in one model. Once trained, it evaluates a new initial condition in a
single forward pass, instead of re-running a numerical solver.

We train a 1D FNO to learn the solution operator of the heat equation
$\partial_t u = \nu \, \partial_{xx} u$ on a periodic domain: given $u(\cdot, 0)$, predict
$u(\cdot, T)$. Because this equation has a closed-form solution in Fourier space, we can
generate an unlimited amount of exact training data without a PDE solver, and score the model
against ground truth exactly.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

N = 64          # grid points on the periodic domain [0, 2*pi)
nu = 0.05       # diffusion coefficient
T_HORIZON = 0.3 # how far forward in time the operator maps

x_grid = np.linspace(0, 2 * np.pi, N, endpoint=False)


def random_initial_conditions(batch, max_mode=8, rng=np.random):
    """Random smooth periodic functions: sums of a few low-frequency sines/cosines."""
    u0 = np.zeros((batch, N))
    for b in range(batch):
        for freq in range(1, max_mode + 1):
            amp = rng.randn() / freq
            phase = rng.rand() * 2 * np.pi
            u0[b] += amp * np.cos(freq * x_grid + phase)
    return u0


def heat_equation_operator(u0_batch, t):
    """Exact solution operator via FFT: each Fourier mode k decays as exp(-nu*k^2*t)."""
    u0_hat = np.fft.rfft(u0_batch, axis=-1)
    k = np.fft.rfftfreq(N, d=1.0 / N)
    decay = np.exp(-nu * (k**2) * t)
    return np.fft.irfft(u0_hat * decay, n=N, axis=-1)

## Sanity-check the data generator against physics

The heat equation conserves total mass (the zero Fourier mode never decays) and always
*dissipates* energy (every other mode strictly decays). If our synthetic data doesn't satisfy
these, something is wrong before we even get to the model.

In [ ]:
rng = np.random.RandomState(0)
u0_check = random_initial_conditions(4, rng=rng)
ut_check = heat_equation_operator(u0_check, T_HORIZON)

print("mean before:", u0_check.mean(axis=1))
print("mean after: ", ut_check.mean(axis=1), "(mass should be conserved)")
print("energy before:", (u0_check**2).sum(axis=1))
print("energy after: ", (ut_check**2).sum(axis=1), "(should be lower - diffusion dissipates energy)")

## The Fourier Neural Operator

Each spectral layer: FFT the input, apply a learned complex-valued weight to only the lowest
`modes` frequencies (discarding the rest - this is the key inductive bias: solution operators
for smooth PDEs are dominated by low frequencies), then inverse FFT back to the grid. A
pointwise (1x1 conv) skip connection is added at every layer, as in the original FNO paper.

In [ ]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        self.modes = modes
        scale = 1 / (in_ch * out_ch)
        self.weight = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes, dtype=torch.cfloat))

    def forward(self, x):
        B, C, L = x.shape
        x_ft = torch.fft.rfft(x, dim=-1)
        out_ft = torch.zeros(B, self.weight.shape[1], x_ft.shape[-1], dtype=torch.cfloat, device=x.device)
        out_ft[:, :, : self.modes] = torch.einsum("bix,iox->box", x_ft[:, :, : self.modes], self.weight)
        return torch.fft.irfft(out_ft, n=L, dim=-1)


class FNOBlock(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.spectral = SpectralConv1d(width, width, modes)
        self.pointwise = nn.Conv1d(width, width, 1)

    def forward(self, x):
        return torch.relu(self.spectral(x) + self.pointwise(x))


class FNO1d(nn.Module):
    def __init__(self, width=32, modes=12, n_blocks=3):
        super().__init__()
        self.lift = nn.Conv1d(1, width, 1)
        self.blocks = nn.ModuleList([FNOBlock(width, modes) for _ in range(n_blocks)])
        self.project = nn.Conv1d(width, 1, 1)

    def forward(self, x):
        x = self.lift(x)
        for block in self.blocks:
            x = block(x)
        return self.project(x)

In [ ]:
n_train, n_test = 256, 64
U0_train = random_initial_conditions(n_train, rng=rng)
UT_train = heat_equation_operator(U0_train, T_HORIZON)
U0_test = random_initial_conditions(n_test, rng=rng)
UT_test = heat_equation_operator(U0_test, T_HORIZON)

X_train = torch.tensor(U0_train, dtype=torch.float32, device=device).unsqueeze(1)
Y_train = torch.tensor(UT_train, dtype=torch.float32, device=device).unsqueeze(1)
X_test = torch.tensor(U0_test, dtype=torch.float32, device=device).unsqueeze(1)
Y_test = torch.tensor(UT_test, dtype=torch.float32, device=device).unsqueeze(1)

model = FNO1d().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(401):
    opt.zero_grad()
    pred = model(X_train)
    loss = torch.mean((pred - Y_train) ** 2)
    loss.backward()
    opt.step()
    if epoch % 100 == 0:
        print(f"epoch {epoch:4d}  train MSE {loss.item():.6f}")

## Evaluate: did it learn the operator, or just copy the input?

The most important check for any operator-learning model: compare against the trivial
"identity" baseline (predict output = input, i.e. learn nothing). If the model isn't
meaningfully better than that, it hasn't learned the dynamics.

In [ ]:
with torch.no_grad():
    pred_test = model(X_test)
    test_mse = torch.mean((pred_test - Y_test) ** 2).item()
    identity_baseline_mse = torch.mean((X_test - Y_test) ** 2).item()

print(f"test MSE (FNO):            {test_mse:.6f}")
print(f"test MSE (identity baseline): {identity_baseline_mse:.6f}")
print(f"improvement over baseline: {identity_baseline_mse / test_mse:.0f}x")

import matplotlib.pyplot as plt

i = 0
plt.figure(figsize=(8, 4))
plt.plot(x_grid, X_test[i, 0].cpu(), label="input u(x, 0)", alpha=0.7)
plt.plot(x_grid, Y_test[i, 0].cpu(), label="true u(x, T)", linewidth=2)
plt.plot(x_grid, pred_test[i, 0].cpu(), "--", label="FNO prediction", linewidth=2)
plt.xlabel("x")
plt.legend()
plt.title("Heat equation: FNO-predicted vs. true evolution")
plt.show()

## Next steps

- Swap `heat_equation_operator` for a PDE without a closed form (e.g. Burgers' equation) and
  generate training data with a numerical solver instead.
- Try a harder `nu` (less diffusion, sharper features) or a shorter/longer `T_HORIZON`.
- Compare with `pinn.ipynb` - a PINN solves *one* instance of an equation; a neural operator
  learns the solution operator across a whole distribution of initial conditions at once.